# Act 1 — Why purpose-built instead of relational?

You have already met relational on RDS and Aurora. They are extraordinarily general — almost any data shape can be coaxed into rows and columns, and SQL can express almost any query. So why does AWS publish a whole zoo of other databases?

Because at scale, generality has a cost. A relational engine handling petabyte-scale analytics, microsecond key-value lookups, graph traversal across a billion edges, or time-series sensor data, will be slower and more expensive than a database purpose-built for that shape. The services in this notebook each answer one specific kind of access pattern well — and answer almost everything else badly. The skill is recognising which cue points at which service.

## The database zoo at a glance

DynamoDB is AWS's serverless NoSQL database — a key-value and document store that delivers single-digit millisecond reads and writes at any scale, with no servers to manage, no schema to define upfront, and automatic replication across three AZs. It anchors most serverless architectures on AWS. Redshift is AWS's data warehouse — a columnar, massively-parallel store built for analytical queries over terabytes-to-petabytes of data. Around the two of them sit a handful of purpose-built databases (Neptune, DocumentDB, Keyspaces, Timestream, MemoryDB) and an analytics stack on top of S3 (Athena, Glue, EMR, OpenSearch) that the rest of this notebook walks through.

The through-line: relational databases are general-purpose tools that have to be coaxed into specific shapes. The services here are purpose-built — each one is the right answer to a specific kind of data or access pattern, and a wrong answer to almost everything else. The skill is recognising which is which.

# Act 2 — DynamoDB: serverless key-value at any scale

When the access pattern is "fetch by ID, fast, at any scale, with no servers to manage", **DynamoDB** is the answer. It anchors most serverless architectures on AWS — pair Lambda with DynamoDB and you have an application that scales from zero requests per second to tens of thousands without changing anything operational.

The core trade is that DynamoDB rewards designing your access patterns first and choosing the keys and indexes to support them. The relational habit of normalising the schema and querying however you want does not transfer. This act walks the model — partitions and hot keys, the two capacity modes, the three read consistency levels, the two flavours of secondary index, the four read operations, **DAX** for microsecond reads, **Streams and Global Tables** for replication, and **TTL plus point-in-time recovery** for operational hygiene.

## DynamoDB — Core Model

DynamoDB is a fully managed NoSQL store. Tables hold **items** (records); items hold **attributes** (fields); items in the same table can have different attributes — DynamoDB is **schemaless** apart from the primary key, which every item must carry.

| Concept | DynamoDB | SQL equivalent |
|---|---|---|
| Container | table | table |
| Record | item | row |
| Field | attribute | column |

Attributes support a rich set of types — strings, numbers, binary, booleans, null, sets of those, lists, and nested maps. A single item is capped at 400 KB.

The primary key uniquely identifies each item, and it comes in two shapes.

- **Simple primary key** — just a **partition key** (PK). Each item has a globally unique PK. Examples: `userId`, `productSku`.
- **Composite primary key** — **partition key plus sort key** (PK + SK). Many items share a PK; the SK orders them and provides uniqueness. Examples: `customerId` + `orderId`, `deviceId` + `timestamp`.

The composite form is the unlock for almost every interesting access pattern in DynamoDB — "get all orders for a customer", "get all readings for a device in a time range". You query by the partition key and use the sort key to filter or order within it.

## Partitions and Hot Keys

Under the hood DynamoDB shards data across many physical partitions. The partition that holds an item is chosen by **hashing the partition key**. Two implications fall out of that, and both bite people in production.

**You want high cardinality on the partition key.** If millions of items share a handful of partition key values — say, `status = "active"` or `country = "US"` — then almost all reads and writes land on the same physical partition. Throughput for that partition is capped, and the table throttles even when total cluster capacity looks healthy. This is the classic **hot partition** failure. The fix is to pick a partition key with many distinct values that traffic spreads across naturally, or to synthesise one (e.g. `status#shardId` with a random shard suffix).

**You can't query across partitions cheaply.** A `Query` operation always targets one partition key value. If you need to fetch items across many partition keys, you either run many parallel queries or fall back to `Scan` — and `Scan` reads the entire table, which is expensive at scale.

The practical takeaway: DynamoDB rewards designing your **access patterns first**, then choosing partition key, sort key, and indexes to support them. The relational habit of "design a normalised schema, query however you want" doesn't transfer.

## Capacity Modes

DynamoDB has two billing modes, and the choice shapes both cost and operational behaviour.

**Provisioned capacity.** You specify **read capacity units** (RCUs) and **write capacity units** (WCUs) upfront. One RCU buys one strongly-consistent read per second of an item up to 4 KB, or two eventually-consistent reads of the same size. One WCU buys one write per second of an item up to 1 KB. Items larger than the base size consume proportionally more units. Provisioned mode is cheaper for steady, predictable workloads — but if traffic exceeds the provisioned ceiling, requests are **throttled**. Auto Scaling sits on top: you set a min/max range and a target utilisation, and DynamoDB scales the provisioned values up or down on a minute-scale.

**On-demand capacity.** No capacity planning — DynamoDB scales instantly with traffic and you pay per request. Costs more per request than provisioned, but never throttles on traffic spikes. The right default for unpredictable workloads, new tables before traffic shape is known, dev and test, and any application where bursting matters more than steady cost.

| | Provisioned | On-demand |
|---|---|---|
| Cost shape | lower for steady traffic | higher per request |
| Capacity planning | required | none |
| Scaling | Auto Scaling, minutes | instant |
| Throttling on spikes? | yes | no |
| Best for | predictable workloads | spiky, unpredictable, new tables |

You can switch between modes — there's a 24-hour cooldown after each switch.

## Read Consistency

DynamoDB writes are committed to multiple AZs before they acknowledge. Reads come in three flavours that trade consistency against cost.

| Read type | Returns | Cost per 4 KB | When to use |
|---|---|---|---|
| **Eventually consistent** (default) | maybe stale, ~1 s lag worst case | 0.5 RCU | default — most reads tolerate slight lag |
| **Strongly consistent** | always the latest committed write | 1 RCU | when you've just written and need to read your own write |
| **Transactional** | ACID across multiple items / tables | 2 RCU (reads), 2 WCU (writes) | financial, inventory, anywhere a multi-item invariant matters |

Strongly consistent reads cost twice an eventually consistent read; transactional reads cost twice that again. The practical pattern: read eventually consistent by default, use strongly consistent only at points where the application logic requires it, and reserve transactions for the small set of operations that genuinely need cross-item atomicity.

## Secondary Indexes

The base table's partition key answers one access pattern well. Secondary indexes let you answer others by maintaining a second copy of items keyed differently.

**Local Secondary Index (LSI)** keeps the base table's partition key but uses a **different sort key**. "Same customer, ordered by date" instead of "same customer, ordered by order ID". Limits worth knowing: LSIs must be defined **at table creation** — you can't add or drop them later. Up to **5 per table**. They share the base table's RCUs and WCUs. They support **strongly consistent reads**. And every partition key in the table is capped at **10 GB** when LSIs are involved.

**Global Secondary Index (GSI)** uses a **different partition key** (and optional different sort key). "All orders with status = SHIPPED, ordered by date." GSIs can be created or dropped **at any time**. Up to **20 per table** by default. They have **separate provisioned throughput** from the base table. They support **eventually consistent reads only** — never strongly consistent. And there's no 10 GB cap.

| | LSI | GSI |
|---|---|---|
| Partition key | same as base table | different |
| When created | table creation only | anytime |
| Strong consistency? | yes | no |
| Throughput | shared with base | separate |
| Per-PK size cap | 10 GB | none |
| Limit per table | 5 | 20 (default) |

The decision rule: if you need strong consistency on the alternate key, it has to be an LSI (and you have to plan it at table-design time). If you need a completely different partition key, or you didn't anticipate the access pattern up front, that's a GSI.

## Operations: Get, Query, Scan, Transactions

Four shapes of read in DynamoDB, with very different cost profiles.

- **`GetItem`** — fetch a single item by primary key. The cheapest, fastest read.
- **`Query`** — fetch items within one partition key value, optionally filtered by sort key range. Efficient — it only reads the relevant partition. This is the workhorse of DynamoDB access patterns.
- **`Scan`** — read every item in the table, then filter. **Charges capacity for the entire table**, regardless of how many items match. Almost always wrong in production. If you find yourself reaching for `Scan`, the access pattern probably wants a GSI instead.
- **`BatchGetItem` / `BatchWriteItem`** — up to 100 gets or 25 writes in one call, across tables. Saves round-trips, not capacity.

Filter expressions deserve a separate note: they apply **after** the read, so the items that get filtered out still cost capacity. They're for trimming the response payload, not for cheap querying.

**Transactions** are DynamoDB's ACID layer. `TransactWriteItems` performs up to 100 puts, updates, deletes, or conditional checks across multiple tables atomically — all succeed or all roll back. `TransactGetItems` does the same for consistent multi-item reads. Cost is roughly **2× the equivalent non-transactional operation**, but the alternative — application-side compensation logic — is usually worse than that.

Condition expressions are the quieter feature worth knowing: `PutItem` and `UpdateItem` can take an `ConditionExpression` that requires some attribute to be in a certain state for the write to succeed. "Insert this order only if no item with this ID exists" or "decrement inventory only if quantity is greater than zero" — both single-call patterns, no transaction needed.

## DAX — DynamoDB Accelerator

DAX is an in-memory cache purpose-built for DynamoDB. It sits between the application and the table, delivers **microsecond reads** on cache hits (vs single-digit milliseconds for DynamoDB itself), and is drop-in compatible with the DynamoDB SDK — you swap the client and the existing code works unchanged.

Three properties worth knowing.

- **Write-through.** Writes go to DynamoDB first, then update the DAX cache on success. Reads after a write see the new value immediately on the DAX node that handled the write, with brief lag on other nodes.
- **Item cache + query cache.** Two separate caches with independent TTLs — five minutes for items, one minute for queries by default. `GetItem` reads hit the item cache; `Query` and `Scan` results hit the query cache.
- **VPC-only, multi-AZ.** A DAX cluster runs as nodes inside your VPC with private IPs, spread across AZs for HA.

DAX is the right answer when a DynamoDB workload is read-heavy on a hot subset, latency below a millisecond matters, and you don't want application logic to manage a cache. It only works in front of DynamoDB. For caching arbitrary data (computed aggregations, third-party API responses, anything not DynamoDB-backed) you're back to ElastiCache.

## Streams and Global Tables

**DynamoDB Streams** is a 24-hour change log of item-level modifications. Every create, update, and delete on a table can be captured as a stream record, ordered per partition key. You choose what each record contains: keys only, new image, old image, or both old and new.

Lambda consumes streams via an event source mapping — exactly-once delivery, scaled automatically to the number of shards. Streams are the foundation under a lot of DynamoDB patterns:

- **Event-driven downstream work** — fan a write out to email, push notification, downstream service
- **Search indexing** — replicate writes to OpenSearch for full-text search
- **Audit logging** — append every change to S3 via Firehose
- **Materialised views** — maintain a derived projection in another DynamoDB table
- **Replication** — the basis of Global Tables

**Global Tables** is DynamoDB's multi-region, **multi-active** replication. Every region accepts both reads and writes; changes propagate to every other region typically in **under a second**. Conflict resolution is **last-writer-wins** based on timestamp — there's no application-level reconciliation. Requirements: Streams must be enabled on the table, and the table must be on-demand mode or provisioned with Auto Scaling.

The use cases are global applications that need local-latency writes everywhere, active-active DR, and data residency where users in a region read and write to a regional copy. The trade-off is the last-writer-wins model — workloads that can't tolerate occasional silent overwrites need either application-level conflict handling or a different architecture.

## TTL and PITR

Two small features that matter operationally.

**TTL (Time to Live)** auto-deletes items after a per-item expiry timestamp. Enable TTL on the table, pick an attribute holding a Unix epoch in seconds, and DynamoDB removes expired items in the background — typically within 48 hours of the expiry. **TTL deletes don't consume WCUs** and they show up in Streams, so cleanup logic can hang off them. Important caveat: items aren't deleted at exactly the TTL time; they can still appear in reads until the background sweep runs, so filter on the expiry attribute if strict expiry matters.

**PITR (point-in-time recovery)** is continuous backup. With PITR enabled, you can restore the table to any second within the last **35 days**, recovering a new table at that timestamp. PITR is the standard guard against application bugs that corrupt data or human DELETE-without-WHERE accidents. **On-demand backups** are the manual companion — a full snapshot you trigger and keep until you delete it, with cross-region copy supported.

In [ ]:
import boto3
from boto3.dynamodb.conditions import Key, Attr
from decimal import Decimal

dynamodb = boto3.resource("dynamodb", region_name="us-east-1")
client = boto3.client("dynamodb", region_name="us-east-1")

# Composite-key table with a GSI for status lookups, on-demand, Streams on for downstream Lambda
dynamodb.create_table(
    TableName="Orders",
    KeySchema=[
        {"AttributeName": "customerId", "KeyType": "HASH"},
        {"AttributeName": "orderId",    "KeyType": "RANGE"},
    ],
    AttributeDefinitions=[
        {"AttributeName": "customerId", "AttributeType": "S"},
        {"AttributeName": "orderId",    "AttributeType": "S"},
        {"AttributeName": "status",     "AttributeType": "S"},
        {"AttributeName": "orderDate",  "AttributeType": "S"},
    ],
    GlobalSecondaryIndexes=[{
        "IndexName": "StatusIndex",
        "KeySchema": [
            {"AttributeName": "status",    "KeyType": "HASH"},
            {"AttributeName": "orderDate", "KeyType": "RANGE"},
        ],
        "Projection": {"ProjectionType": "ALL"},
    }],
    BillingMode="PAY_PER_REQUEST",
    StreamSpecification={"StreamEnabled": True, "StreamViewType": "NEW_AND_OLD_IMAGES"},
)

orders = dynamodb.Table("Orders")

# Conditional put — fail if an order with this PK+SK already exists
orders.put_item(
    Item={
        "customerId": "cust-001",
        "orderId":    "order-001",
        "orderDate":  "2026-04-18",
        "status":     "PENDING",
        "amount":     Decimal("59.99"),
    },
    ConditionExpression="attribute_not_exists(orderId)",
)

# Query by partition key, with a sort-key prefix — efficient, only reads the relevant partition
resp = orders.query(
    KeyConditionExpression=Key("customerId").eq("cust-001") & Key("orderId").begins_with("order-"),
    ScanIndexForward=False,         # newest first
)

# Query the GSI — answers "all pending orders today" without scanning
orders.query(
    IndexName="StatusIndex",
    KeyConditionExpression=Key("status").eq("PENDING") & Key("orderDate").eq("2026-04-18"),
)

# Transaction — decrement inventory and create the order atomically
client.transact_write_items(TransactItems=[
    {"Update": {
        "TableName": "Inventory",
        "Key": {"sku": {"S": "ABC"}},
        "UpdateExpression": "SET quantity = quantity - :q",
        "ConditionExpression": "quantity >= :q",
        "ExpressionAttributeValues": {":q": {"N": "2"}},
    }},
    {"Put": {
        "TableName": "Orders",
        "Item": {
            "customerId": {"S": "cust-002"}, "orderId": {"S": "order-002"},
            "status": {"S": "CONFIRMED"}, "amount": {"N": "29.99"},
        },
    }},
])

# Act 3 — Redshift and the analytics stack

DynamoDB and the relational engines are for transactional workloads — many small queries against current state. Analytics is a different shape entirely. Few large queries. Full scans over billions of rows. Aggregations and joins across years of history. Run those queries on a transactional engine and they grind.

The right tool is a columnar, massively parallel data warehouse, and on AWS that is **Redshift**. Around Redshift sit four other services that complete the picture — **Athena** for serverless SQL over S3, **Glue** for ETL and the table catalogue, **EMR** for custom Hadoop and Spark workloads, **OpenSearch** for full-text search and log analytics. This act walks Redshift's architecture, how data gets in (COPY and Spectrum), the two physical-design choices that make or break performance, and then each of the satellite services in turn.

## Redshift — Data Warehouse for OLAP

Redshift is AWS's fully managed data warehouse. The job is **OLAP** — Online Analytical Processing — which means big queries over big datasets: monthly revenue rollups, customer cohort analysis, multi-table joins across years of history. The opposite of OLTP, the transactional workload that RDS and Aurora are built for.

| | OLTP (RDS / Aurora) | OLAP (Redshift) |
|---|---|---|
| Queries | many small, point lookups | few large, full scans + aggregations |
| Data size | GB to low TB | TB to PB |
| Users | the application | analysts and BI tools |
| Storage | row-oriented | **columnar** |

The key word is **columnar**. A row store keeps each row's fields together on disk — perfect for fetching one whole row by primary key. A columnar store keeps each column's values together — perfect for `SELECT SUM(revenue) FROM orders` because only the revenue column gets read off disk. For an analytical query that touches three columns out of fifty across a billion rows, columnar storage cuts I/O by an order of magnitude or more. Compression is also dramatically better on columnar data because values within a column are usually similar.

That one design choice is most of why Redshift is fast on analytics and unsuited to transactional workloads.

## Redshift Architecture

A Redshift cluster has a **leader node** and a set of **compute nodes**. The leader receives SQL from clients, parses and plans the query, and distributes work to the compute nodes. Each compute node executes its slice in parallel and ships intermediate results back. This is **MPP** — massively parallel processing — and it's what lets Redshift chew through terabytes by horizontally distributing the work.

Two node families.

- **RA3** — managed storage. Compute and storage scale independently; storage lives in S3 under the hood with a local SSD cache for hot data. You add or remove compute nodes without touching the data. The recommended default for any new cluster.
- **DC2** — local SSD storage tied to the node. Smaller datasets, lower cost at small scale, but storage and compute scale together. Mostly legacy now.

**Redshift Serverless** removes the cluster concept entirely. You pick a base capacity in **RPUs** (Redshift Processing Units), and Redshift scales compute up and down automatically as queries run. Pay per RPU-hour. Right for intermittent analytics, dev/test, and workloads that don't justify a permanently-on cluster.

**Multi-AZ** is available on RA3 clusters — the cluster spans two AZs with automatic failover. Because RA3 stores data in S3, failover is fast and doesn't require any data replication step.

## Loading Data — COPY and Spectrum

The right way to bulk-load Redshift is the **`COPY`** command — it reads files in parallel from S3, with each compute node slice pulling its own files. Supports CSV, JSON, Parquet, ORC, Avro; orders of magnitude faster than `INSERT` statements. The pattern in almost every Redshift pipeline is: land data in S3, then `COPY` it into the warehouse.

**Redshift Spectrum** is the other half of the S3 story — it lets Redshift query data **directly in S3** without loading it into the cluster first. You define external tables backed by S3 objects (usually via the **AWS Glue Data Catalog**), and standard Redshift SQL can join those external tables against tables that live inside the cluster. Spectrum spins up its own pool of nodes to scan S3 in parallel, billed per terabyte scanned.

The split is useful: hot, frequently-joined data lives inside Redshift; cold historical data stays in S3 and gets queried via Spectrum when needed. The same SQL works against both — the planner decides what Spectrum reads and what the cluster reads.

One networking detail worth knowing: **Enhanced VPC Routing** forces `COPY` and `UNLOAD` traffic between Redshift and S3 to traverse your VPC (via S3 gateway endpoints) rather than the public internet. Required for compliance setups where data must never touch the public path.

## Distribution and Sort Keys

Two physical design choices govern Redshift performance on large tables.

**Distribution style** decides how rows are placed across compute nodes. Pick badly and queries shuffle huge amounts of data across the network during joins; pick well and joins are local to each node.

| Style | How | When |
|---|---|---|
| **AUTO** | Redshift chooses based on table size | the default — start here |
| **KEY** | hash a column; equal values land on the same node | large tables joined frequently on that column |
| **ALL** | full copy of the table on every node | small dimension tables joined a lot |
| **EVEN** | round-robin | no clear join key |

A classic pattern: a large `orders` fact table distributed by `customer_id` (KEY), the small `customers` dimension table distributed ALL. The join then runs locally on each node — no network shuffle.

**Sort keys** decide the order data is stored within each node. Redshift keeps zone maps recording the min and max value of the sort key in each storage block, so queries with a `WHERE` clause on the sort key **skip blocks entirely** instead of scanning them. The right sort key turns a full-table scan into a small range read. Common picks are a date column (most queries filter by recent time) or the column most often used in `WHERE` clauses.

Get these two right and a query can go from minutes to seconds with no schema or SQL change.

## Around Redshift — Athena, Glue, EMR, OpenSearch

Four services that round out the analytics stack alongside Redshift.

**Athena.** Serverless SQL over data in S3. No infrastructure, no loading — point Athena at S3 objects (via the Glue Data Catalog), write SQL, pay per terabyte scanned. The pattern is **data lake querying** when a full warehouse is overkill: log analysis, ad-hoc analyst queries, dashboards over Parquet in S3. Athena and Redshift Spectrum look similar from the outside; the distinction is that Spectrum is part of a Redshift cluster (and uses cluster pricing), while Athena is fully standalone. Use Athena when you don't already have Redshift; use Spectrum when you do.

**Glue.** Managed ETL plus the **Glue Data Catalog**, the metastore for tables in S3. Glue crawlers scan S3 and infer table schemas into the catalog, which Athena, Spectrum, and EMR all read from. Glue ETL jobs are serverless Spark — pay per DPU-second to run extract-transform-load pipelines. The catalog is the genuinely important part; the ETL feature competes with managed Spark on EMR.

**EMR.** Managed Hadoop ecosystem — Spark, Hive, Presto, HBase, Flink on EC2 (or EKS or Serverless). Used when teams want the full open-source big-data toolkit with control over cluster sizing and library versions. Heavier ops than Athena or Glue, but the most flexibility for arbitrary Spark or Hive workloads.

**OpenSearch.** Managed Elasticsearch / OpenSearch — search and log analytics over JSON documents. The default fit is full-text search and log analytics dashboards (the OpenSearch successor to the ELK stack). Often paired with DynamoDB Streams or Kinesis Firehose to keep an OpenSearch index in sync with operational data.

The shape: Athena and Spectrum query S3 directly with SQL; Glue runs ETL and catalogues; EMR runs heavyweight Spark workloads; OpenSearch handles search and log dashboards; Redshift is the heavy-duty SQL warehouse for analytical workloads that justify it.

In [ ]:
import boto3

redshift = boto3.client("redshift", region_name="us-east-1")
data_api = boto3.client("redshift-data", region_name="us-east-1")

# RA3 cluster — storage decoupled from compute, Enhanced VPC Routing for COPY/UNLOAD
redshift.create_cluster(
    ClusterIdentifier="prod-warehouse",
    NodeType="ra3.xlplus",
    NumberOfNodes=2,
    MasterUsername="admin",
    MasterUserPassword="use-secrets-manager-instead",
    DBName="analytics",
    ClusterSubnetGroupName="redshift-subnet-group",
    VpcSecurityGroupIds=["sg-0redshift"],
    Encrypted=True,
    EnhancedVpcRouting=True,
    AutomatedSnapshotRetentionPeriod=7,
)

# Async SQL via the Data API — no persistent connection, IAM-authed, fits Lambda well
resp = data_api.execute_statement(
    ClusterIdentifier="prod-warehouse",
    Database="analytics",
    SecretArn="arn:aws:secretsmanager:us-east-1:111122223333:secret:redshift-creds",
    Sql="""
        SELECT date_trunc('month', order_date) AS month,
               SUM(revenue)                    AS total_revenue,
               COUNT(DISTINCT customer_id)     AS unique_customers
        FROM   fact_orders
        WHERE  order_date >= '2026-01-01'
        GROUP  BY 1
        ORDER  BY 1
    """,
)

# Spectrum — query S3 data directly via an external schema backed by Glue
spectrum_setup = """
    CREATE EXTERNAL SCHEMA s3_logs
    FROM DATA CATALOG
    DATABASE 'logs_db'
    IAM_ROLE 'arn:aws:iam::111122223333:role/RedshiftSpectrumRole';

    SELECT user_id, COUNT(*) AS hits
    FROM   s3_logs.access_log              -- external table backed by Parquet in S3
    JOIN   dim_users USING (user_id)       -- regular Redshift dimension
    WHERE  request_time >= '2026-04-01'
    GROUP  BY user_id
    ORDER  BY hits DESC
    LIMIT  100;
"""

# Act 4 — Purpose-built: the rest of the database zoo

A handful of databases exist because both relational and DynamoDB are wrong shapes for specific workloads. Each one is a one-liner away from being unfit for everything else — the value is in recognising the cue and jumping to the service.

This act covers six — **DocumentDB** for MongoDB workloads, **Neptune** for graph data, **Keyspaces** for Cassandra, **Timestream** for time-series, **MemoryDB** for Redis as a durable primary store, and **QLDB** for cryptographically verifiable audit trails. The pattern is cue-to-service, not service-to-capability.

## Purpose-Built Databases

A handful of databases that exist because relational and DynamoDB are wrong shapes for specific workloads. Each one is a one-liner away from being unfit for everything else — the value is in recognising the cue.

**DocumentDB.** MongoDB-compatible document store. Same Aurora-style storage underneath (six copies across three AZs, auto-grow to 128 TB, up to 15 read replicas). Use it for MongoDB workloads you're migrating to AWS or for content / catalog data with variable structure. Cue: *MongoDB*, *document database*.

**Neptune.** Fully managed graph database. Models data as **nodes and edges** and runs graph traversal queries via Gremlin, openCypher, or SPARQL. The fit is highly connected data — social graphs, fraud detection (link analysis across accounts, devices, IPs), recommendation engines, knowledge graphs, IT topology. Cue: *graph*, *social network*, *fraud*, *friend-of-friend*, *interconnected*.

**Keyspaces.** Serverless Cassandra-compatible store, speaks CQL. Use it to migrate existing Cassandra workloads or for wide-row high-write workloads (IoT telemetry, time-series-ish). Cue: *Cassandra*.

**Timestream.** Purpose-built time-series database. Auto-tiers recent data in memory and historical data on magnetic storage, with built-in time-series functions (interpolation, smoothing, approximation). Cue: *IoT*, *time-series*, *sensor data*, *metrics over time*.

**MemoryDB for Redis.** Redis API on top of a **durable** Multi-AZ transaction log. The distinction from ElastiCache Redis is durability — every write is committed to the log before it's acknowledged, so MemoryDB is safe as the **primary database**, not just a cache. Reads still take microseconds; writes take single-digit milliseconds (the log commit). Cue: *durable Redis*, *Redis as primary store*, *can't afford to lose this data*.

**QLDB.** Ledger database with a cryptographically verifiable **immutable journal** — every change is hashed and appended, and you can mathematically prove no record was altered. Serverless, queried in PartiQL. The fit is centralised audit trails: financial ledgers, supply-chain provenance, healthcare records. Cue: *immutable audit log*, *cryptographically verifiable history*. If the scenario requires multi-party decentralised trust instead, that's **Managed Blockchain**, not QLDB.

The selection skill is the cue → service jump. The cues above are the ones the SAA exam and most real architectures lean on.

# Act 5 — Picking the right database

You now have RDS and Aurora from the previous notebook, plus everything in this one. The decision matrix below collapses the choice into a cue-to-service lookup. Real architectures rarely use just one — they end up stacking two or three together. DynamoDB plus OpenSearch for operational state with search. Redshift plus Athena plus Glue for a warehouse fed by a lake. DynamoDB plus DAX for hot reads. The question is rarely "which database" — it is "which mix".

## Picking the Right Database

A cue-to-service cheat sheet for the services in this notebook and the relational chapter. Match the cue, get the service.

| Cue | Service |
|---|---|
| key-value / document at any scale, single-digit ms, serverless | **DynamoDB** |
| DynamoDB but microseconds and read-heavy | **DAX** |
| MongoDB-compatible documents | **DocumentDB** |
| Cassandra-compatible | **Keyspaces** |
| graph — social, fraud, recommendations, friend-of-friend | **Neptune** |
| time-series, IoT, sensors, metrics over time | **Timestream** |
| Redis as a durable primary store | **MemoryDB** |
| immutable cryptographic audit trail (centralised) | **QLDB** |
| multi-party decentralised ledger | **Managed Blockchain** |
| OLAP, BI, TB–PB analytics in SQL | **Redshift** |
| Redshift querying S3 alongside cluster data | **Redshift Spectrum** |
| serverless SQL on S3 with no warehouse | **Athena** |
| serverless Spark ETL + table catalogue | **AWS Glue** |
| managed Hadoop ecosystem — Spark, Hive, Presto, HBase | **EMR** |
| full-text search and log analytics | **OpenSearch** |

The meta-pattern: each service is the answer to one specific data shape or access pattern. Real architectures end up with two or three of them stacked — DynamoDB plus OpenSearch for operational + search, Redshift plus Athena plus Glue for warehouse + lake, DynamoDB plus DAX for hot reads. The choice isn't "which database" but "which mix".